# Santec TSL-570 Tunable Semiconductor Laser

This notebook demonstrates the QCoDeS driver for the Santec TSL-570 tunable laser.

**Key points**
- Uses SCPI command mode via the driver
- Wavelength/frequency are in SI units (`m`, `Hz`)
- Power setpoint is in `mW`
- Uses a VISA resource address (for example TCPIP socket)

Update the resource string before running.

## 1. Import and Connect

In [1]:
import time

from qcodes_contrib_drivers.drivers.Santec.Santec_TSL570 import SantecTSL570

# Example VISA resource for LAN socket
resource = "TCPIP0::192.168.50.29::5000::SOCKET"
laser = SantecTSL570("laser", resource)
print(f"Connected to: {laser.get_idn()}")

NoTagError: `git describe --long --dirty --always --tags '--match=v*'` could not find a tag


Connected to: SANTEC TSL-570 (serial:21090055, firmware:0026.0026.0011) in 0.30s
Connected to: {'vendor': 'SANTEC', 'model': 'TSL-570', 'serial': '21090055', 'firmware': '0026.0026.0011'}


## 2. System Information

In [2]:
print("=== System Information ===")
print(f"Firmware version:  {laser.system_version()}")
print(f"Product code:      {laser.system_code()}")
print(f"Command set:       {laser.command_set()}")
print(f"Interlock status:  {laser.system_lock()}")
print(f"System alert:      {laser.system_alert()}")
print(f"Error queue:       {laser.system_error()}")

=== System Information ===
Firmware version:  0026.0026.0011
Product code:      A-500630-P-F-AP-00-1
Command set:       SCPI
Interlock status:  False
System alert:      No alerts.
Error queue:       {'code': 0, 'message': 'No error'}


## 3. Basic Wavelength and Power Control

In [3]:
print("=== Current Settings ===")
print(f"Wavelength: {laser.wavelength() * 1e9:.4f} nm ({laser.wavelength():.10e} m)")
print(f"Frequency:  {laser.frequency() / 1e12:.6f} THz ({laser.frequency():.10e} Hz)")
print(f"Power:      {laser.power():.4f} mW")
print(f"Output:     {laser.output()}")
print(f"Shutter:    {laser.shutter()}")

target_wavelength_nm = 1550.0
laser.wavelength(target_wavelength_nm * 1e-9)
print(f"\nWavelength set to: {laser.wavelength() * 1e9:.4f} nm")

target_frequency_thz = 193.5
laser.frequency(target_frequency_thz * 1e12)
print(f"Frequency set to: {laser.frequency() / 1e12:.4f} THz")
print(f"Corresponding wavelength: {laser.wavelength() * 1e9:.4f} nm")

laser.power_unit("mW")
laser.power(1.0)
print(f"Power set to: {laser.power():.4f} mW")

=== Current Settings ===
Wavelength: 1537.3972 nm (1.5373972000e-06 m)
Frequency:  195.000000 THz (1.9500000000e+14 Hz)
Power:      3.0100 mW
Output:     False
Shutter:    True

Wavelength set to: 1550.0000 nm
Frequency set to: 193.5000 THz
Corresponding wavelength: 1549.3150 nm
Power set to: 1.0000 mW


## 4. Fine Wavelength Control

In [4]:
laser.wavelength_fine(25.0)
print(f"Fine tuning offset: {laser.wavelength_fine()}")

laser.wavelength_offset(0.01e-9)
print(f"Wavelength offset: {laser.wavelength_offset() * 1e9:.3f} nm")

laser.disable_fine_tuning()
print("Fine-tuning disabled")

Fine tuning offset: 0.0
Wavelength offset: 0.010 nm
Fine-tuning disabled


## 5. Wavelength and Frequency Sweep Configuration

In [5]:
start_nm = 1530.0
stop_nm = 1570.0
laser.sweep_start_wavelength(start_nm * 1e-9)
laser.sweep_stop_wavelength(stop_nm * 1e-9)

start_thz = 187.3
stop_thz = 196.1
laser.sweep_start_frequency(start_thz * 1e12)
laser.sweep_stop_frequency(stop_thz * 1e12)

laser.sweep_mode(1)
laser.sweep_speed(10)
laser.sweep_step_wavelength(0.1e-9)
laser.sweep_dwell(0.1)
laser.sweep_cycles(1)
laser.sweep_delay(0.5)

print("=== Sweep Parameters ===")
print(f"Wavelength start: {laser.sweep_start_wavelength() * 1e9:.2f} nm")
print(f"Wavelength stop:  {laser.sweep_stop_wavelength() * 1e9:.2f} nm")
print(f"Frequency start:  {laser.sweep_start_frequency() / 1e12:.2f} THz")
print(f"Frequency stop:   {laser.sweep_stop_frequency() / 1e12:.2f} THz")
print(f"Mode:             {laser.sweep_mode()} (0=step 1-way, 1=cont 1-way, 2=step 2-way, 3=cont 2-way)")
print(f"Speed:            {laser.sweep_speed()} nm/s")
print(f"Step (wavelength): {laser.sweep_step_wavelength() * 1e9:.3f} nm")
print(f"Step (frequency):  {laser.sweep_step_frequency() / 1e9:.3f} GHz")
print(f"Dwell:            {laser.sweep_dwell()} s")
print(f"Cycles:           {laser.sweep_cycles()}")
print(f"Delay:            {laser.sweep_delay()} s")

=== Sweep Parameters ===
Wavelength start: 1600.60 nm
Wavelength stop:  1528.77 nm
Frequency start:  187.30 THz
Frequency stop:   196.10 THz
Mode:             1 (0=step 1-way, 1=cont 1-way, 2=step 2-way, 3=cont 2-way)
Speed:            10.0 nm/s
Step (wavelength): 0.100 nm
Step (frequency):  1.000 GHz
Dwell:            0.1 s
Cycles:           1
Delay:            0.5 s


## 6. Trigger Configuration

In [6]:
laser.trigger_input_external(True)
laser.trigger_input_polarity("RISE")  # RISE or FALL
laser.trigger_input_standby(False)

laser.trigger_output_timing("STEP")  # NONE, STOP, START, STEP
laser.trigger_output_polarity("RISE")  # RISE or FALL
laser.trigger_output_step(1e-12)
laser.trigger_output_setting("WAVELENGTH")  # WAVELENGTH or TIME

print("=== Trigger Input ===")
print(f"External: {laser.trigger_input_external()}")
print(f"Polarity: {laser.trigger_input_polarity()}")
print(f"Standby:  {laser.trigger_input_standby()}")

print("\n=== Trigger Output ===")
print(f"Timing:   {laser.trigger_output_timing()}")
print(f"Polarity: {laser.trigger_output_polarity()}")
print(f"Step:     {laser.trigger_output_step() * 1e12:.3f} pm")
print(f"Mode:     {laser.trigger_output_setting()}")

=== Trigger Input ===
External: True
Polarity: RISE
Standby:  False

=== Trigger Output ===
Timing:   STEP
Polarity: RISE
Step:     1.000 pm
Mode:     WAVELENGTH


## 7. Execute and Monitor Sweep

In [7]:
print(f"Sweep state before start: {laser.sweep_state()}")
laser.sweep_single()

print("=== Monitoring Sweep ===")
for idx in range(10):
    state = laser.sweep_state()
    wl_nm = laser.wavelength() * 1e9
    count = laser.sweep_count()
    print(f"[{idx + 1}] State: {state:16s} lambda = {wl_nm:8.3f} nm  Count: {count}")
    if state == "STOPPED":
        break
    time.sleep(0.2)

laser.sweep_stop()
print(f"Final state: {laser.sweep_state()}")

Sweep state before start: STOPPED
=== Monitoring Sweep ===
[1] State: STOPPED          lambda = 1549.315 nm  Count: 0
Final state: STOPPED


## 8. Snapshot and Logged Data

In [8]:
# Avoid update=True because sweep_range_minimum_wavelength / sweep_range_maximum_wavelength can time out on some firmware
snapshot = laser.snapshot(update=False)

print("=== Snapshot (selected parameters) ===")
params = snapshot["parameters"]
for name in ["wavelength", "frequency", "power", "output", "sweep_mode", "sweep_state"]:
    if name in params:
        print(f"{name:20s}: {params[name].get('value', 'N/A')}")

print("\nLogged points:", laser.readout_points())
# Data readouts return numpy arrays
# wavelength_log = laser.readout_data()
# power_log = laser.readout_power_data()

=== Snapshot (selected parameters) ===
wavelength          : 1.549315e-06
frequency           : 193500000000000.0
power               : 1.0
output              : False
sweep_mode          : 1
sweep_state         : STOPPED

Logged points: 0


## 9. Cleanup and Shutdown

In [9]:
print("Shutting down laser output...")
laser.sweep_stop()
laser.shutter(False)
laser.output(False)
print(f"Sweep:   {laser.sweep_state()}")
print(f"Output:  {laser.output()}")
print(f"Shutter: {laser.shutter()}")

laser.close()
print("Connection closed.")

Shutting down laser output...
Sweep:   STOPPED
Output:  False
Shutter: True
Connection closed.
